# f6_m04a_stress.ipynb

## Qué hace
Evalúa la robustez del modelo CatBoost ante perturbaciones controladas.
Simula 4 escenarios adversos: ruido gaussiano en features numéricas,
valores extremos (outliers), features ausentes (imputadas con la mediana)
y degradación gradual de la calidad de los datos.
Mide cómo cambian AUC y F1 bajo cada perturbación.

## Requisitos
- `data/05_modelado/X_test_prep.parquet`
- `data/05_modelado/y_test.parquet`
- `data/05_modelado/models/CatBoost__balanced.pkl`

## Genera
- `results/fase6/stress_resultados.parquet`
- `results/fase6/stress_ruido.png`
- `results/fase6/stress_outliers.png`
- `results/fase6/stress_missing.png`
- `docs/html/fase6/m04a_stress.html`

## Flujo
Cargar datos → baseline → perturbación ruido → perturbación outliers →
perturbación missing → gráficos → HTML

## Siguiente
`f6_m04b_calibracion.ipynb` — calibración de probabilidades

In [1]:
# 1. CONFIGURACIÓN DE RUTAS
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError('No se encontró src/ subiendo desde ' + str(start))

ROOT = _encontrar_root(Path.cwd())
sys.path.insert(0, str(ROOT))

DIR_DATA    = ROOT / 'data' / '05_modelado'
DIR_MODELS  = ROOT / 'data' / '05_modelado' / 'models'
DIR_RESULTS = ROOT / 'results' / 'fase6'
DIR_RESULTS.mkdir(parents=True, exist_ok=True)
DIR_HTML = ROOT / 'docs' / 'html' / 'fase6'
DIR_HTML.mkdir(parents=True, exist_ok=True)

print(f'ROOT:        {ROOT}')
print(f'DIR_RESULTS: {DIR_RESULTS}')

ROOT:        C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI
DIR_RESULTS: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\results\fase6


In [2]:
# 2. IMPORTS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.metrics import roc_auc_score, f1_score
from src.html.render import render_pagina_desde_fichero

plt.rcParams['figure.dpi'] = 120
RNG = np.random.default_rng(42)

print('Imports OK.')
from src.config_entorno import NOMBRES_LEGIBLES_FEATURES

def nombre_legible(f): return NOMBRES_LEGIBLES_FEATURES.get(f, f.replace("_"," "))


Imports OK.


In [3]:
# 3. CARGAR DATOS Y MODELO
# Pipeline completo para predict_proba — no extraer named_steps aquí.

X_test_prep  = pd.read_parquet(DIR_DATA / 'X_test_prep.parquet')
y_test       = pd.read_parquet(DIR_DATA / 'y_test.parquet').squeeze()
pipeline_cat = joblib.load(DIR_MODELS / 'CatBoost__balanced.pkl')

y_true = y_test.values.ravel()
features_num = X_test_prep.select_dtypes(include=[np.number]).columns.tolist()
features_cat = X_test_prep.select_dtypes(exclude=[np.number]).columns.tolist()

# Baseline
y_prob_base = pipeline_cat.predict_proba(X_test_prep)[:, 1]
y_pred_base = (y_prob_base >= 0.5).astype(int)
auc_base    = roc_auc_score(y_true, y_prob_base)
f1_base     = f1_score(y_true, y_pred_base)

print(f'Baseline — AUC: {auc_base:.4f} | F1: {f1_base:.4f}')
print(f'Features numéricas: {len(features_num)}')
print(f'Features categóricas: {len(features_cat)}')

Baseline — AUC: 0.9530 | F1: 0.8188
Features numéricas: 27
Features categóricas: 0


In [4]:
# 4. STRESS TEST 1 — RUIDO GAUSSIANO
# Añadimos ruido gaussiano proporcional a la desviación estándar de cada feature.
# Niveles: 5%, 10%, 20%, 30%, 50% del std de cada feature.
# Simula errores de medición o datos de baja calidad.

niveles_ruido = [0.05, 0.10, 0.20, 0.30, 0.50]
resultados_ruido = []

stds = X_test_prep[features_num].std()

for nivel in niveles_ruido:
    X_pert = X_test_prep.copy()
    ruido  = RNG.normal(0, nivel, size=X_pert[features_num].shape) * stds.values
    X_pert[features_num] = X_pert[features_num] + ruido

    y_prob = pipeline_cat.predict_proba(X_pert)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    auc    = roc_auc_score(y_true, y_prob)
    f1     = f1_score(y_true, y_pred)
    resultados_ruido.append({'nivel': nivel, 'auc': auc, 'f1': f1})
    print(f'Ruido {nivel*100:.0f}% std — AUC: {auc:.4f} | F1: {f1:.4f}')

df_ruido = pd.DataFrame(resultados_ruido)

Ruido 5% std — AUC: 0.9524 | F1: 0.8157
Ruido 10% std — AUC: 0.9505 | F1: 0.8145
Ruido 20% std — AUC: 0.9408 | F1: 0.7931
Ruido 30% std — AUC: 0.9285 | F1: 0.7786
Ruido 50% std — AUC: 0.9023 | F1: 0.7339


In [5]:
# 5. STRESS TEST 2 — VALORES EXTREMOS (OUTLIERS)
# Sustituimos un porcentaje de observaciones por valores extremos
# (media ± 3*std). Simula errores de entrada de datos o casos atípicos.

pcts_outliers = [0.01, 0.05, 0.10, 0.20, 0.30]
resultados_outliers = []

medias = X_test_prep[features_num].mean()

for pct in pcts_outliers:
    X_pert = X_test_prep.copy()
    n_pert = int(len(X_pert) * pct)
    idx_pert = RNG.choice(len(X_pert), size=n_pert, replace=False)

    # Valores extremos: media + 3*std con signo aleatorio
    signos = RNG.choice([-1, 1], size=(n_pert, len(features_num)))
    X_pert.iloc[idx_pert, X_pert.columns.get_indexer(features_num)] = (
        medias.values + signos * 3 * stds.values
    )

    y_prob = pipeline_cat.predict_proba(X_pert)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    auc    = roc_auc_score(y_true, y_prob)
    f1     = f1_score(y_true, y_pred)
    resultados_outliers.append({'pct': pct, 'auc': auc, 'f1': f1})
    print(f'Outliers {pct*100:.0f}% obs — AUC: {auc:.4f} | F1: {f1:.4f}')

df_outliers = pd.DataFrame(resultados_outliers)

Outliers 1% obs — AUC: 0.9498 | F1: 0.8163
Outliers 5% obs — AUC: 0.9251 | F1: 0.7974
Outliers 10% obs — AUC: 0.9007 | F1: 0.7757
Outliers 20% obs — AUC: 0.8429 | F1: 0.7195
Outliers 30% obs — AUC: 0.8119 | F1: 0.6853


Outliers 20% obs — AUC: 0.8791 | F1: 0.6822


Outliers 30% obs — AUC: 0.8443 | F1: 0.6284


In [6]:
# 6. STRESS TEST 3 — FEATURES AUSENTES (MISSING)
# Sustituimos features con la mediana (imputación simple).
# Probamos eliminar de 1 a 5 features importantes según SHAP.
# Si no existe el ranking SHAP, usamos las primeras features numéricas.

RUTA_SHAP = DIR_RESULTS / 'shap_importancia_comparativa.parquet'
if RUTA_SHAP.exists():
    df_shap_imp  = pd.read_parquet(RUTA_SHAP)
    features_top = df_shap_imp.nsmallest(5, 'rank_medio').index.tolist()
    features_top = [f for f in features_top if f in features_num]
else:
    features_top = features_num[:5]

resultados_missing = []
medianas = X_test_prep[features_num].median()

for n_missing in range(1, len(features_top) + 1):
    feats_eliminar = features_top[:n_missing]
    X_pert = X_test_prep.copy()
    X_pert[feats_eliminar] = medianas[feats_eliminar].values

    y_prob = pipeline_cat.predict_proba(X_pert)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    auc    = roc_auc_score(y_true, y_prob)
    f1     = f1_score(y_true, y_pred)
    resultados_missing.append({
        'n_features': n_missing,
        'features':   ', '.join(feats_eliminar),
        'auc':        auc,
        'f1':         f1
    })
    print(f'Missing {n_missing} feat — AUC: {auc:.4f} | F1: {f1:.4f} | {feats_eliminar}')

df_missing = pd.DataFrame(resultados_missing)

Missing 1 feat — AUC: 0.9077 | F1: 0.7169 | ['cred_superados_anio_1er']
Missing 2 feat — AUC: 0.8897 | F1: 0.6782 | ['cred_superados_anio_1er', 'n_anios_beca']
Missing 3 feat — AUC: 0.8580 | F1: 0.5783 | ['cred_superados_anio_1er', 'n_anios_beca', 'n_anios_trabajando']
Missing 4 feat — AUC: 0.8249 | F1: 0.5773 | ['cred_superados_anio_1er', 'n_anios_beca', 'n_anios_trabajando', 'n_anios_sin_notas']
Missing 5 feat — AUC: 0.7626 | F1: 0.5664 | ['cred_superados_anio_1er', 'n_anios_beca', 'n_anios_trabajando', 'n_anios_sin_notas', 'nota_1er_anio']


In [7]:
# 7. GUARDAR RESULTADOS Y GRÁFICOS

# Guardar todos los resultados
df_stress = pd.concat([
    df_ruido.assign(test='ruido'),
    df_outliers.assign(test='outliers'),
    df_missing.assign(test='missing')
], ignore_index=True)
df_stress.to_parquet(DIR_RESULTS / 'stress_resultados.parquet')

# Gráfico ruido
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metrica in zip(axes, ['auc', 'f1']):
    baseline = auc_base if metrica == 'auc' else f1_base
    ax.plot([n*100 for n in df_ruido['nivel']], df_ruido[metrica],
            marker='o', color='#3182ce', linewidth=2)
    ax.axhline(baseline, color='gray', linestyle='--', linewidth=1, label=f'Baseline ({baseline:.3f})')
    ax.set_xlabel('Nivel de ruido (% std)')
    ax.set_ylabel(metrica.upper())
    ax.set_title(f'{metrica.upper()} bajo ruido gaussiano')
    ax.legend(fontsize=9)
plt.suptitle('Stress test — Ruido gaussiano en features numéricas', fontsize=12)
plt.tight_layout()
ruta_ruido = DIR_RESULTS / 'stress_ruido.png'
plt.savefig(ruta_ruido, dpi=120, bbox_inches='tight')
plt.close()

# Gráfico outliers
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metrica in zip(axes, ['auc', 'f1']):
    baseline = auc_base if metrica == 'auc' else f1_base
    ax.plot([p*100 for p in df_outliers['pct']], df_outliers[metrica],
            marker='o', color='#e53e3e', linewidth=2)
    ax.axhline(baseline, color='gray', linestyle='--', linewidth=1, label=f'Baseline ({baseline:.3f})')
    ax.set_xlabel('% observaciones con outliers')
    ax.set_ylabel(metrica.upper())
    ax.set_title(f'{metrica.upper()} bajo outliers')
    ax.legend(fontsize=9)
plt.suptitle('Stress test — Valores extremos (media ± 3σ)', fontsize=12)
plt.tight_layout()
ruta_outliers = DIR_RESULTS / 'stress_outliers.png'
plt.savefig(ruta_outliers, dpi=120, bbox_inches='tight')
plt.close()

# Gráfico missing
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metrica in zip(axes, ['auc', 'f1']):
    baseline = auc_base if metrica == 'auc' else f1_base
    ax.plot(df_missing['n_features'], df_missing[metrica],
            marker='o', color='#dd6b20', linewidth=2)
    ax.axhline(baseline, color='gray', linestyle='--', linewidth=1, label=f'Baseline ({baseline:.3f})')
    ax.set_xlabel('Nº features top eliminadas (imputadas con mediana)')
    ax.set_ylabel(metrica.upper())
    ax.set_title(f'{metrica.upper()} bajo missing top features')
    ax.legend(fontsize=9)
plt.suptitle('Stress test — Features ausentes (imputación con mediana)', fontsize=12)
plt.tight_layout()
ruta_missing = DIR_RESULTS / 'stress_missing.png'
plt.savefig(ruta_missing, dpi=120, bbox_inches='tight')
plt.close()

print('Gráficos guardados.')
# Gráfico degradación relativa (% pérdida vs baseline) — CUM LAUDE
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (df_plot, x_col, x_label, titulo_plot) in zip(axes, [
    (df_ruido,    [n*100 for n in df_ruido['nivel']],   'Nivel ruido (% std)', 'Ruido gaussiano'),
    (df_outliers, [p*100 for p in df_outliers['pct']], '% obs con outliers',   'Valores extremos'),
]):
    pct_auc = (auc_base - df_plot['auc']) / auc_base * 100
    pct_f1  = (f1_base  - df_plot['f1'])  / f1_base  * 100
    ax.plot(x_col, pct_auc, marker='o', color='#3182ce', linewidth=2, label='% pérdida AUC')
    ax.plot(x_col, pct_f1,  marker='s', color='#e53e3e', linewidth=2, label='% pérdida F1')
    ax.axhline(5, color='#d69e2e', linestyle='--', linewidth=1, alpha=0.7, label='Umbral 5%')
    ax.set_xlabel(x_label)
    ax.set_ylabel('% degradación vs baseline')
    ax.set_title(f'Degradación relativa — {titulo_plot}')
    ax.legend(fontsize=9)
    ax.set_ylim(bottom=0)

plt.suptitle('Fase 6 — Degradación relativa (% pérdida respecto al baseline)', fontsize=12)
plt.tight_layout()
ruta_degradacion = DIR_RESULTS / 'stress_degradacion_relativa.png'
plt.savefig(ruta_degradacion, dpi=120, bbox_inches='tight')
plt.close()
print('✅ Gráfico degradación relativa guardado.')


Gráficos guardados.
✅ Gráfico degradación relativa guardado.


In [8]:
# 8. GENERAR HTML
import base64

def img_b64(ruta: Path) -> str:
    if not ruta.exists():
        return ''
    with open(ruta, 'rb') as f:
        return base64.b64encode(f.read()).decode()

def bloque_imagen(b64: str, titulo: str, caption: str) -> str:
    if not b64:
        return f'<p style="color:#e53e3e">Imagen no disponible: {titulo}</p>'
    return (
        '<div style="margin:24px 0">'
        f'<h3 style="color:#2d3748; font-size:15px">{titulo}</h3>'
        f'<img src="data:image/png;base64,{b64}" style="max-width:100%; border-radius:6px; box-shadow:0 2px 8px rgba(0,0,0,.1)">'
        f'<p style="color:#718096; font-size:12px; margin-top:6px">{caption}</p>'
        '</div>'
    )

# Tabla resumen de degradación máxima
degradacion_auc_ruido    = auc_base - df_ruido['auc'].min()
degradacion_f1_ruido     = f1_base  - df_ruido['f1'].min()
degradacion_auc_outliers = auc_base - df_outliers['auc'].min()
degradacion_f1_outliers  = f1_base  - df_outliers['f1'].min()
degradacion_auc_missing  = auc_base - df_missing['auc'].min()
degradacion_f1_missing   = f1_base  - df_missing['f1'].min()

filas_resumen = (
    f'<tr><td style="padding:8px 12px">Ruido gaussiano (50% std)</td>'
    f'<td style="padding:8px 12px; text-align:center">{degradacion_auc_ruido:.4f}</td>'
    f'<td style="padding:8px 12px; text-align:center">{degradacion_f1_ruido:.4f}</td></tr>'
    f'<tr style="background:#f7fafc"><td style="padding:8px 12px">Outliers (30% observaciones)</td>'
    f'<td style="padding:8px 12px; text-align:center">{degradacion_auc_outliers:.4f}</td>'
    f'<td style="padding:8px 12px; text-align:center">{degradacion_f1_outliers:.4f}</td></tr>'
    f'<tr><td style="padding:8px 12px">Missing top 5 features</td>'
    f'<td style="padding:8px 12px; text-align:center">{degradacion_auc_missing:.4f}</td>'
    f'<td style="padding:8px 12px; text-align:center">{degradacion_f1_missing:.4f}</td></tr>'
)

contenido = (
    '<h2 style="color:#2d3748">Fase 6 — Stress Testing: Robustez del Modelo</h2>'
    '<p style="color:#4a5568; font-size:14px; max-width:800px">'
    'Evaluación de la robustez del modelo CatBoost ante perturbaciones controladas. '
    'Se simulan tres escenarios adversos: ruido gaussiano en las features numéricas, '
    'presencia de valores extremos y ausencia de las features más importantes. '
    'Un modelo robusto mantiene su rendimiento incluso con datos de baja calidad.'
    '</p>'
    f'<p style="color:#4a5568; font-size:13px">'
    f'Baseline — AUC: <strong>{auc_base:.4f}</strong> | F1: <strong>{f1_base:.4f}</strong>'
    f'</p>'
    '<h3 style="color:#2d3748; margin-top:20px">Degradación máxima por escenario</h3>'
    '<table style="width:60%; border-collapse:collapse; font-size:13px; margin-bottom:24px">'
    '<thead><tr style="background:#edf2f7">'
    '<th style="padding:8px 12px; text-align:left">Escenario</th>'
    '<th style="padding:8px 12px; text-align:center">ΔAUC</th>'
    '<th style="padding:8px 12px; text-align:center">ΔF1</th>'
    '</tr></thead>'
    f'<tbody>{filas_resumen}</tbody></table>'
    + bloque_imagen(img_b64(ruta_ruido),
        'AUC y F1 bajo ruido gaussiano',
        'Degradación progresiva del rendimiento al añadir ruido proporcional '
        'a la desviación estándar de cada feature numérica.')
    + bloque_imagen(img_b64(ruta_outliers),
        'AUC y F1 bajo valores extremos',
        'Impacto de sustituir un porcentaje creciente de observaciones '
        'por valores extremos (media ± 3σ).')
    + bloque_imagen(img_b64(ruta_missing),
        'AUC y F1 bajo features ausentes',
        'Degradación al eliminar progresivamente las features más importantes '
        '(imputadas con la mediana). Muestra cuánto depende el modelo de cada feature.')
    + bloque_imagen(img_b64(DIR_RESULTS / 'stress_degradacion_relativa.png'),
    '🏆 Degradación relativa (% pérdida vs baseline)',
    '% de pérdida de AUC y F1 respecto al baseline. '
    'La línea amarilla marca el umbral del 5% — degradaciones por debajo '
    'indican un modelo robusto apto para producción.')
+ '<div style="margin-top:24px; padding:16px; background:#ebf8ff; '
    + 'border-left:4px solid #3182ce; border-radius:6px; font-size:13px; color:#2c5282">'
    + '<strong>Interpretación:</strong> Una caída de AUC inferior a 0.02 bajo ruido moderado '
    + '(10-20% std) indica un modelo robusto. Caídas superiores a 0.05 sugieren '
    + 'sensibilidad excesiva que debería abordarse antes del despliegue en producción.'
    + '</div>'
)

html_completo = render_pagina_desde_fichero('f6_m04a_stress.ipynb', contenido)
ruta_html = DIR_HTML / 'm04a_stress.html'
ruta_html.parent.mkdir(parents=True, exist_ok=True)
ruta_html.write_text(html_completo, encoding='utf-8')
print(f'HTML generado: {ruta_html}')

HTML generado: C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI\docs\html\fase6\m04a_stress.html
